# Understanding PCA Through Interactive Exploration

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
1. Explain PCA as a **variance maximization** problem
2. Understand the equivalence: **minimizing projection error ⟺ maximizing variance**
3. Visualize how PCA finds optimal projection axes in 2D and 3D
4. Recognize PCA's **sensitivity to outliers** and its implications
5. Connect geometric intuition to mathematical formulation

---

## 📚 Prerequisites

- **Linear Algebra**: Vectors, dot products, orthogonal projections
- **Statistics**: Variance, covariance (basic concepts)
- **Python**: NumPy, Matplotlib (you'll learn by doing!)

---

## 📖 Notebook Structure

| Module | Topic | Approach |
|--------|-------|----------|
| **1** | 2D PCA Intuition | Interactive sliders to explore variance and projections |
| **2** | Outlier Sensitivity | Hands-on experiments adding outliers |
| **3** | Extension to 3D | Visualize ellipsoids and Cartesian projections |
| **4** | Arbitrary Axes | Explore projection onto custom 3D coordinate systems |
| **5** | Summary & Next Steps | Connect to sklearn.PCA and real datasets |

---

## 🔗 Related Notebooks

- **After this notebook**: Try `PCA_CityTemp_Plots.ipynb` for comprehensive PCA diagnostics on real temperature data
- **For advanced topics**: See `ComprehensivePCA.ipynb` for outlier detection, contribution analysis, and validation

---

**Ready?** Let's start building intuition! 🚀

---

# Setup: Import Libraries and Helper Functions

First, let's load the tools we need and define helper functions that will be used throughout the notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, HBox, interactive_output, HTML, Dropdown, Layout
import ipywidgets as widgets
from IPython.display import display
import time

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries loaded successfully!")

✓ Libraries loaded successfully!


In [2]:
# ========================================
# 2D Helper Functions
# ========================================

def generate_rotated_data(angle=30.0, var_parallel=5.0, var_perp=0.5, n_samples=300):
    """
    Generate 2D Gaussian data rotated by 'angle' degrees.
    
    Parameters:
    -----------
    angle : float
        Rotation angle in degrees (0-180)
    var_parallel : float
        Variance along the main axis of the data
    var_perp : float
        Variance perpendicular to the main axis
    n_samples : int
        Number of data points to generate
        
    Returns:
    --------
    data : ndarray of shape (n_samples, 2)
        Generated 2D data points
    """
    # Generate data in canonical orientation
    u = np.sqrt(var_parallel) * np.random.randn(n_samples)
    v = np.sqrt(var_perp) * np.random.randn(n_samples)
    data = np.vstack([u, v])
    
    # Apply rotation
    theta = np.deg2rad(angle)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    return (R @ data).T


def projection_error(data, alpha):
    """
    Compute mean squared error when projecting data onto axis at angle alpha.
    
    The error is the sum of squared distances from each point to its projection.
    Equivalently, it's the variance in the perpendicular direction.
    
    Parameters:
    -----------
    data : ndarray of shape (n, 2)
        2D data points
    alpha : float
        Projection axis angle in degrees
        
    Returns:
    --------
    error : float
        Mean squared projection error
    """
    a_rad = np.deg2rad(alpha)
    # Perpendicular vector to projection axis
    perp_vec = np.array([-np.sin(a_rad), np.cos(a_rad)])
    # Coordinates in perpendicular direction
    coords = data @ perp_vec
    return np.mean(coords**2)


def iterative_search(data, step_size=5):
    """
    Greedy search algorithm to find axis that minimizes projection error.
    
    This simulates how PCA finds the optimal direction through iterative improvement.
    
    Parameters:
    -----------
    data : ndarray of shape (n, 2)
        2D data points
    step_size : float
        Step size in degrees for search
        
    Returns:
    --------
    trajectory : list of tuples
        Each tuple contains (angle, error) for each iteration
    """
    current_alpha = 0
    trajectory = []
    
    while True:
        err = projection_error(data, current_alpha)
        left = projection_error(data, (current_alpha - step_size) % 180)
        right = projection_error(data, (current_alpha + step_size) % 180)
        trajectory.append((current_alpha, err))
        
        # Check if we found a local minimum
        if err <= left and err <= right:
            break
            
        # Move in direction of lower error
        if left < right:
            current_alpha = (current_alpha - step_size) % 180
        else:
            current_alpha = (current_alpha + step_size) % 180
            
    return trajectory

print("✓ 2D helper functions defined!")

✓ 2D helper functions defined!


In [3]:
# ========================================
# 3D Helper Functions
# ========================================

def generate_3d_data(var_x=5.0, var_y=2.0, var_z=1.0, rot_x=0, rot_y=0, rot_z=0, n_samples=400):
    """
    Generate rotated 3D Gaussian ellipsoid data.
    
    Parameters:
    -----------
    var_x, var_y, var_z : float
        Variances along each principal axis
    rot_x, rot_y, rot_z : float
        Euler angles for rotation (in degrees)
    n_samples : int
        Number of data points
        
    Returns:
    --------
    data : ndarray of shape (n_samples, 3)
        Generated 3D data points
    """
    # Generate data in canonical orientation
    u = np.sqrt(var_x) * np.random.randn(n_samples)
    v = np.sqrt(var_y) * np.random.randn(n_samples)
    w = np.sqrt(var_z) * np.random.randn(n_samples)
    data = np.vstack([u, v, w]).T

    # Apply Euler rotations (X->Y->Z order)
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(np.deg2rad(rot_x)), -np.sin(np.deg2rad(rot_x))],
                   [0, np.sin(np.deg2rad(rot_x)),  np.cos(np.deg2rad(rot_x))]])
    
    Ry = np.array([[ np.cos(np.deg2rad(rot_y)), 0, np.sin(np.deg2rad(rot_y))],
                   [0, 1, 0],
                   [-np.sin(np.deg2rad(rot_y)), 0, np.cos(np.deg2rad(rot_y))]])
    
    Rz = np.array([[np.cos(np.deg2rad(rot_z)), -np.sin(np.deg2rad(rot_z)), 0],
                   [np.sin(np.deg2rad(rot_z)),  np.cos(np.deg2rad(rot_z)), 0],
                   [0, 0, 1]])
    
    R = Rz @ Ry @ Rx
    return data @ R.T


def project_to_plane(data, axis1, axis2):
    """
    Project 3D data onto a 2D plane defined by two axes.
    
    Parameters:
    -----------
    data : ndarray of shape (n, 3)
        3D data points
    axis1, axis2 : ndarray of shape (3,)
        Two orthogonal unit vectors defining the projection plane
        
    Returns:
    --------
    coords1, coords2 : ndarray of shape (n,)
        Coordinates in the 2D projection plane
    """
    coords1 = data @ axis1
    coords2 = data @ axis2
    return coords1, coords2


def euler_axes(alpha, beta, gamma):
    """
    Compute three orthogonal axes based on Euler angle rotations.
    
    This allows us to explore projection onto arbitrary coordinate systems.
    
    Parameters:
    -----------
    alpha, beta, gamma : float
        Euler angles in degrees
        
    Returns:
    --------
    ax_alpha, ax_beta, ax_gamma : ndarray of shape (3,)
        Three orthogonal unit vectors
    """
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(np.deg2rad(alpha)), -np.sin(np.deg2rad(alpha))],
                   [0, np.sin(np.deg2rad(alpha)),  np.cos(np.deg2rad(alpha))]])
    
    Ry = np.array([[ np.cos(np.deg2rad(beta)), 0, np.sin(np.deg2rad(beta))],
                   [0, 1, 0],
                   [-np.sin(np.deg2rad(beta)), 0, np.cos(np.deg2rad(beta))]])
    
    Rz = np.array([[np.cos(np.deg2rad(gamma)), -np.sin(np.deg2rad(gamma)), 0],
                   [np.sin(np.deg2rad(gamma)),  np.cos(np.deg2rad(gamma)), 0],
                   [0, 0, 1]])
    
    R = Rz @ Ry @ Rx
    # Return the three column vectors (the new axes)
    return R[:,0], R[:,1], R[:,2]

print("✓ 3D helper functions defined!")

✓ 3D helper functions defined!


---

# Module 1: 2D PCA Intuition

## 🧮 Mathematical Foundation

### Projection onto an axis

Given a data point $\mathbf{x}$ and a unit vector $\mathbf{u}$, the **projection** of $\mathbf{x}$ onto $\mathbf{u}$ is:

$$\text{proj}_{\mathbf{u}}(\mathbf{x}) = (\mathbf{x} \cdot \mathbf{u})\mathbf{u}$$

### Projection Error

The **error** (distance from point to projection) is:

$$e = \|\mathbf{x} - \text{proj}_{\mathbf{u}}(\mathbf{x})\|^2$$

### Key Insight: Variance-Error Duality

**PCA finds the direction that maximizes variance along the projection axis.**

This is equivalent to **minimizing the perpendicular error**:

$$\max_{\mathbf{u}} \text{Var}(\text{proj}_{\mathbf{u}}(\mathbf{X})) \quad \Longleftrightarrow \quad \min_{\mathbf{u}} \sum_i \|\mathbf{x}_i - \text{proj}_{\mathbf{u}}(\mathbf{x}_i)\|^2$$

**Why?** Because total variance is constant: $\text{Var}_{\parallel} + \text{Var}_{\perp} = \text{Total Var}$

---

## 🎮 Interactive Exploration

Use the sliders below to explore how data orientation, variance, and projection axes interact.

In [ ]:
def plot_2d_pca(angle=30.0, var_parallel=5.0, var_perp=0.5, alpha=0.0):
    """
    Visualize 2D PCA: data scatter, projections, and error curve.
    
    Left panel: Scatter plot with projection lines
    Right panel: Projection error as function of axis angle
    """
    # Generate data
    data = generate_rotated_data(angle, var_parallel, var_perp)
    x, y = data[:,0], data[:,1]

    # Compute projection vectors
    alpha_rad = np.deg2rad(alpha)
    axis_vec = np.array([np.cos(alpha_rad), np.sin(alpha_rad)])
    perp_vec = np.array([-np.sin(alpha_rad), np.cos(alpha_rad)])

    # Project points onto both axes
    coords_on_axis = data @ axis_vec
    coords_on_perp = data @ perp_vec
    proj_points_alpha = np.outer(coords_on_axis, axis_vec)
    proj_points_perp = np.outer(coords_on_perp, perp_vec)
    proj_error = np.mean(coords_on_perp**2)

    # Compute error curve for all angles
    alphas = np.linspace(0, 180, 181)
    errors = [projection_error(data, a) for a in alphas]

    # Create dual-panel plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [2.5, 1]})
    
    # Left panel: Scatter + projections
    axes[0].scatter(x, y, alpha=0.5, c="blue", s=40, label="Original points")
    axes[0].scatter(proj_points_alpha[:,0], proj_points_alpha[:,1], 
                   alpha=0.8, c="red", s=40, label=f"Proj α={alpha:.1f}°")
    axes[0].scatter(proj_points_perp[:,0], proj_points_perp[:,1], 
                   alpha=0.8, c="green", s=40, label="Proj ⊥ axis")

    # Draw sample error lines
    for i in np.linspace(0, len(x)-1, 20, dtype=int):
        axes[0].plot([x[i], proj_points_alpha[i,0]], [y[i], proj_points_alpha[i,1]], 
                    "r--", alpha=0.3)
        axes[0].plot([x[i], proj_points_perp[i,0]], [y[i], proj_points_perp[i,1]], 
                    "g--", alpha=0.3)

    # Draw projection axes
    line_len = max(np.max(np.abs(x)), np.max(np.abs(y))) * 1.7
    axes[0].plot([-line_len*axis_vec[0], line_len*axis_vec[0]], 
                [-line_len*axis_vec[1], line_len*axis_vec[1]], 
                "r--", lw=2, label="α-axis")
    axes[0].plot([-line_len*perp_vec[0], line_len*perp_vec[0]], 
                [-line_len*perp_vec[1], line_len*perp_vec[1]], 
                "g--", lw=2, label="⊥ α-axis")
    
    axes[0].axhline(0, color="k", lw=1, alpha=0.5)
    axes[0].axvline(0, color="k", lw=1, alpha=0.5)
    axes[0].set_aspect("equal")
    axes[0].set_title(f"Projection at α={alpha:.1f}°\nError={proj_error:.2f}")
    axes[0].legend()

    # Right panel: Error curve
    axes[1].plot(alphas, errors, label="Error vs α")
    axes[1].axvline(alpha, color="r", linestyle="--", label=f"α={alpha:.1f}°")
    axes[1].set_xlabel("α (degrees)")
    axes[1].set_ylabel("Projection error (MSE)")
    axes[1].set_title("Projection error curve")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

# Create sliders
angle_slider = FloatSlider(min=0, max=180, step=5, value=30, description="Data θ")
var_parallel_slider = FloatSlider(min=0.1, max=10, step=0.1, value=5, description="Var ∥")
var_perp_slider = FloatSlider(min=0.0, max=5, step=0.1, value=0.5, description="Var ⟂")
alpha_slider = FloatSlider(min=0, max=180, step=5, value=0, description="Axis α")

# Link sliders to plot
out = interactive_output(plot_2d_pca, {
    'angle': angle_slider,
    'var_parallel': var_parallel_slider,
    'var_perp': var_perp_slider,
    'alpha': alpha_slider
})

ui = HBox([out, VBox([angle_slider, var_parallel_slider, var_perp_slider, alpha_slider])])
display(ui)

## 📝 Guided Exploration Tasks

Work through these steps to build your PCA intuition:

### Step 1: Explore data orientation
- Use the **"Data θ"** slider to rotate the data cloud
- Observe how the cloud aligns with the axes
- **Question:** When θ = 0°, which axis captures most variance?

<details>
<summary>💡 Click for answer</summary>
The x-axis (horizontal) captures most variance when θ = 0° because the data is aligned with it.
</details>

---

### Step 2: Adjust variances
- Set **Var ⟂ = 0**. What happens to the error curve?
- Make **Var ⟂** very large. How does the data shape change?

<details>
<summary>💡 Click for answer</summary>
When Var ⟂ = 0, the data becomes a perfect line and the error curve has a sharp minimum. When Var ⟂ is large, the data becomes more circular and the error curve flattens (all directions are similarly good).
</details>

---

### Step 3: Find the optimal projection
- Move the **"Axis α"** slider to find where error is minimum
- Watch the vertical red line in the error curve plot
- **Question:** Does the optimal α match the data orientation θ?

<details>
<summary>💡 Click for answer</summary>
Yes! The optimal α should equal θ (or θ + 90°, since they represent the same line). PCA automatically finds this optimal direction.
</details>

---

### Step 4: Understand the duality
- Notice: **red projections** (on α-axis) spread out when error is low
- Notice: **green projections** (perpendicular) cluster together when error is low
- **Key insight:** Minimizing error = Maximizing spread (variance)

---

## ✅ Self-Check Quiz

**Q1:** If you project data onto a random axis, what happens to the error?
- ☐ Always increases
- ☑ Usually increases (unless you're lucky)
- ☐ Stays the same

**Q2:** PCA finds the projection axis that:
- ☑ Minimizes reconstruction error
- ☑ Maximizes variance
- ☐ Minimizes variance
- ☑ Aligns with data orientation

**Q3:** When Var ⟂ = 0 (data is 1D), how many good projection directions are there?
- ☐ None
- ☐ One
- ☑ Two (θ and θ + 180°, same line)
- ☐ Infinite

<details>
<summary>📊 Check your understanding</summary>

**Answers:**
- Q1: Usually increases — only the optimal direction minimizes error
- Q2: All three are correct! These are equivalent formulations of PCA
- Q3: Two — but they represent the same line, just opposite directions
</details>

---

# Module 2: PCA and Outlier Sensitivity

## 🎯 Why This Matters

PCA is based on **variance** (squared distances), which means **outliers can drastically affect results**.

**Mathematical reason:** Variance formula includes $x^2$, so extreme points contribute quadratically.

$$\text{Var}(X) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

A single point far from the center can dominate the sum!

---

## 🧪 Interactive Experiment

Use the widget below to add outliers and observe how PCA's optimal axis changes.

In [5]:
# Generate initial data
data0 = generate_rotated_data(40, 5, 0.5)
data_outliers = data0.copy()

# Pre-compute error curve
alphas = np.linspace(0, 180, 181)
errors = [projection_error(data_outliers, a) for a in alphas]

# Create Plotly figure
fig = go.FigureWidget()

# Left panel: Scatter plot
fig.add_scatter(
    x=data_outliers[:,0].tolist(), 
    y=data_outliers[:,1].tolist(), 
    mode="markers", 
    marker=dict(color="blue", size=6), 
    name="Data"
)
fig.add_scatter(x=[], y=[], mode="lines", line=dict(color="red", dash="dash"), name="α-axis")
fig.add_scatter(x=[], y=[], mode="lines", line=dict(color="green", dash="dash"), name="⊥ axis")

# Right panel: Error curve
fig.add_scatter(
    x=alphas.tolist(), 
    y=[float(v) for v in errors], 
    mode="lines", 
    line=dict(color="gray"), 
    name="Error curve", 
    xaxis="x2", 
    yaxis="y2"
)
fig.add_scatter(x=[], y=[], mode="lines", line=dict(color="red", dash="dash"), name="Current α", xaxis="x2", yaxis="y2")
fig.add_scatter(x=[], y=[], mode="markers", marker=dict(color="red", size=10), name="Current error", xaxis="x2", yaxis="y2")

fig.update_layout(
    title="Add outliers and click 'Recompute' to see PCA adapt",
    width=1200, 
    height=600,
    xaxis=dict(domain=[0, 0.45], title="x", scaleanchor="y", scaleratio=1),
    yaxis=dict(title="y"),
    xaxis2=dict(domain=[0.55, 1], title="α (degrees)"),
    yaxis2=dict(title="Projection error (MSE)")
)

# Widgets
x_input = widgets.FloatText(description="x:", value=0.0)
y_input = widgets.FloatText(description="y:", value=0.0)
add_btn = widgets.Button(description="Add point", button_style="info")
recompute_btn = widgets.Button(description="Recompute", button_style="success")
reset_btn = widgets.Button(description="Reset", button_style="danger")

def add_point(b):
    """Add a new point to the dataset."""
    global data_outliers
    new_point = np.array([[x_input.value, y_input.value]])
    data_outliers = np.vstack([data_outliers, new_point])
    
    with fig.batch_update():
        fig.data[0].x = data_outliers[:,0].tolist()
        fig.data[0].y = data_outliers[:,1].tolist()
    
    fig.layout.title = f"Added point ({x_input.value:.2f}, {y_input.value:.2f}). Press 'Recompute' to update PCA."

def recompute(b):
    """Recompute optimal PCA direction using iterative search."""
    global data_outliers
    
    # Recompute error curve
    all_errors = [projection_error(data_outliers, a) for a in alphas]
    
    # Find optimal direction
    trajectory = iterative_search(data_outliers)
    
    # Animate the search
    L = float(np.max(np.abs(data_outliers)) * 1.5)
    for step, (a, e) in enumerate(trajectory):
        theta = np.deg2rad(a)
        axis_vec = np.array([np.cos(theta), np.sin(theta)])
        perp_vec = np.array([-np.sin(theta), np.cos(theta)])
        
        with fig.batch_update():
            fig.data[1].x = [-L*axis_vec[0], L*axis_vec[0]]
            fig.data[1].y = [-L*axis_vec[1], L*axis_vec[1]]
            fig.data[2].x = [-L*perp_vec[0], L*perp_vec[0]]
            fig.data[2].y = [-L*perp_vec[1], L*perp_vec[1]]
            fig.data[3].x = alphas.tolist()
            fig.data[3].y = [float(v) for v in all_errors]
            fig.data[4].x = [a, a]
            fig.data[4].y = [0, float(max(all_errors))*1.05]
            fig.data[5].x = [a]
            fig.data[5].y = [float(e)]
            fig.layout.title = f"Step {step+1}: α={a:.1f}°, Error={e:.2f}"
        
        time.sleep(0.3)

def reset_plot(b):
    """Reset to original data."""
    global data_outliers
    data_outliers = data0.copy()
    all_errors = [projection_error(data_outliers, a) for a in alphas]
    
    with fig.batch_update():
        fig.data[0].x = data_outliers[:,0].tolist()
        fig.data[0].y = data_outliers[:,1].tolist()
        fig.data[1].x = []
        fig.data[1].y = []
        fig.data[2].x = []
        fig.data[2].y = []
        fig.data[3].x = alphas.tolist()
        fig.data[3].y = [float(v) for v in all_errors]
        fig.data[4].x = []
        fig.data[4].y = []
        fig.data[5].x = []
        fig.data[5].y = []
        fig.layout.title = "Reset to original data."

add_btn.on_click(add_point)
recompute_btn.on_click(recompute)
reset_btn.on_click(reset_plot)

ui_plotly = widgets.HBox([x_input, y_input, add_btn, recompute_btn, reset_btn])
display(ui_plotly, fig)

FigureWidget({
    'data': [{'marker': {'color': 'blue', 'size': 6},
              'mode': 'markers',
              'name': 'Data',
              'type': 'scatter',
              'uid': '81e6729e-031e-4b10-a24f-b96a1f797052',
              'x': [1.1290973799490354, -1.4008221444038034, 1.4765067681079405,
                    ..., 0.000192455076199749, 1.964888990455732,
                    -2.019379930277529],
              'y': [1.2877336446575771, -1.5385055917895145, 1.2654695482579899,
                    ..., 0.510144162723883, 1.8554243331993183,
                    -0.43527428845931493]},
             {'line': {'color': 'red', 'dash': 'dash'},
              'mode': 'lines',
              'name': 'α-axis',
              'type': 'scatter',
              'uid': 'c72b2b00-51b7-485b-81f4-8d26d0310d12',
              'x': [],
              'y': []},
             {'line': {'color': 'green', 'dash': 'dash'},
              'mode': 'lines',
              'name': '⊥ axis',
              't

## 🧪 Structured Experiments

### Experiment 1: Baseline (no outliers)
1. Click **Recompute** to see the optimal axis for the original data
2. **Observation:** The red α-axis should align with the main spread

---

### Experiment 2: Single horizontal outlier
1. Add a point far along the x-axis: `x=50, y=0`
2. Click **Recompute**
3. **Question:** How much did one point change the PCA direction?

<details>
<summary>💡 What to expect</summary>
The optimal axis will rotate toward the outlier because PCA tries to capture the large variance it introduces.
</details>

---

### Experiment 3: Vertical outlier
1. Press **Reset**
2. Add `x=0, y=50`
3. **Recompute**
4. **Question:** Does PCA now prefer the vertical direction?

---

### Experiment 4: Cluster of outliers
1. **Reset**
2. Add 3-4 points around `(40, 40)`
3. **Recompute**
4. **Question:** Do clustered outliers have more or less effect than a single extreme point?

---

### Experiment 5: Balanced extremes
1. **Reset**
2. Add `(50, 0)` and `(-50, 0)`
3. **Recompute**
4. **Question:** Why does PCA still align horizontally?

<details>
<summary>💡 Answer</summary>
Balanced outliers on opposite sides increase variance symmetrically along that axis, reinforcing the horizontal direction.
</details>

---

## 🛡️ Robustness Strategies

PCA is **not robust** to outliers. If you encounter outliers in practice, consider:

1. **Preprocessing:** Remove or downweight outliers before PCA
2. **Robust PCA:** Use methods like:
   - Influence function-based PCA
   - Median-based covariance estimation
   - Sparse PCA (assumes outliers are sparse)
3. **Regularization:** Add constraints to prevent over-fitting to extremes

**Further reading:** See `PCA_CityTemp_Plots.ipynb` for outlier detection methods (Hotelling's T², Q-statistic)

---

# Module 3: Extension to 3D

## 🚀 From 2D to 3D: What Changes?

In 2D, PCA finds **1 optimal direction** (a line).

In 3D, PCA finds **3 orthogonal directions** (principal components):
1. **PC1:** Direction of maximum variance
2. **PC2:** Direction of maximum variance *perpendicular to PC1*
3. **PC3:** Direction perpendicular to both PC1 and PC2

### Mathematical Connection

PCA solves the eigenvalue problem on the **covariance matrix**:

$$\mathbf{C} \mathbf{v}_i = \lambda_i \mathbf{v}_i$$

Where:
- $\mathbf{C} = \frac{1}{n}\mathbf{X}^T\mathbf{X}$ is the covariance matrix
- $\mathbf{v}_i$ are eigenvectors (principal components)
- $\lambda_i$ are eigenvalues (variances along each PC)

---

## 📊 Visualizing 3D Data

Use the sliders below to:
- Control the **shape** of the 3D ellipsoid (Var X, Y, Z)
- **Rotate** the ellipsoid (Rot X, Y, Z)
- View **projections** onto Cartesian planes (XY, YZ, ZX)

In [6]:
def plot_3d_and_projections(var_x=5.0, var_y=2.0, var_z=1.0, rot_x=20, rot_y=10, rot_z=0):
    """
    Visualize 3D Gaussian data and its 2D Cartesian projections.
    """
    # Generate 3D data
    data = generate_3d_data(var_x, var_y, var_z, rot_x, rot_y, rot_z)
    X, Y, Z = data[:,0], data[:,1], data[:,2]
    L = np.max(np.abs(data)) * 1.2

    # 3D scatter plot
    fig3d = go.Figure(data=[go.Scatter3d(
        x=X, y=Y, z=Z, 
        mode='markers',
        marker=dict(size=3, color='blue', opacity=0.6)
    )])
    
    fig3d.update_layout(
        scene=dict(
            xaxis=dict(range=[-L, L], title="X"),
            yaxis=dict(range=[-L, L], title="Y"),
            zaxis=dict(range=[-L, L], title="Z"),
            aspectmode='cube'
        ),
        title="3D Scatter Plot",
        width=700,
        height=600
    )
    fig3d.show()

    # 2D projections onto Cartesian planes
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    
    # XY plane
    axes[0].scatter(X, Y, alpha=0.5, c="blue", s=10)
    axes[0].set_xlim(-L, L)
    axes[0].set_ylim(-L, L)
    axes[0].set_aspect("equal")
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Y")
    axes[0].set_title("Projection: XY plane")
    axes[0].grid(True, alpha=0.3)

    # YZ plane
    axes[1].scatter(Y, Z, alpha=0.5, c="blue", s=10)
    axes[1].set_xlim(-L, L)
    axes[1].set_ylim(-L, L)
    axes[1].set_aspect("equal")
    axes[1].set_xlabel("Y")
    axes[1].set_ylabel("Z")
    axes[1].set_title("Projection: YZ plane")
    axes[1].grid(True, alpha=0.3)

    # ZX plane
    axes[2].scatter(Z, X, alpha=0.5, c="blue", s=10)
    axes[2].set_xlim(-L, L)
    axes[2].set_ylim(-L, L)
    axes[2].set_aspect("equal")
    axes[2].set_xlabel("Z")
    axes[2].set_ylabel("X")
    axes[2].set_title("Projection: ZX plane")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Create sliders
varx_slider = FloatSlider(min=0.1, max=10, step=0.1, value=5, description="Var X")
vary_slider = FloatSlider(min=0.1, max=10, step=0.1, value=2, description="Var Y")
varz_slider = FloatSlider(min=0.1, max=10, step=0.1, value=1, description="Var Z")
rotx_slider = FloatSlider(min=0, max=180, step=5, value=20, description="Rot X")
roty_slider = FloatSlider(min=0, max=180, step=5, value=10, description="Rot Y")
rotz_slider = FloatSlider(min=0, max=180, step=5, value=0, description="Rot Z")

out_3d = interactive_output(
    plot_3d_and_projections,
    {
        "var_x": varx_slider, 
        "var_y": vary_slider, 
        "var_z": varz_slider,
        "rot_x": rotx_slider, 
        "rot_y": roty_slider, 
        "rot_z": rotz_slider
    }
)

ui_3d = VBox([
    HTML("<h3>3D Data Visualization</h3>"),
    HBox([varx_slider, vary_slider, varz_slider]),
    HBox([rotx_slider, roty_slider, rotz_slider]),
    out_3d
])

display(ui_3d)

## 🎯 3D Exploration Tasks

### Task 1: Understand the ellipsoid shape
- Set `Var X = 5, Var Y = 2, Var Z = 1` with no rotation
- **Observation:** The ellipsoid is stretched along the X-axis
- **Meaning:** X has the most variance (would be PC1)

---

### Task 2: Effect of rotation
- Rotate using `Rot X = 45°`
- **Question:** Which projections change the most?

<details>
<summary>💡 Answer</summary>
YZ and ZX projections change because rotation around X-axis affects Y and Z coordinates.
</details>

---

### Task 3: Equal variances (spherical data)
- Set `Var X = Var Y = Var Z = 3`
- **Question:** What do the projections look like?
- **Insight:** When data is spherical, **all directions are equally good** — PCA becomes arbitrary!

---

### Task 4: Dimension reduction intuition
- Set `Var X = 8, Var Y = 2, Var Z = 0.1`
- **Question:** Could we ignore the Z dimension without losing much information?

<details>
<summary>💡 PCA Insight</summary>
Yes! PCA would identify that PC3 (aligned with Z) contributes very little variance. We could project onto just PC1-PC2 plane and retain ~99% of variance.
</details>

---

# Module 4: Arbitrary Projection Axes (Advanced)

## 🧭 Beyond Cartesian Coordinates

So far, we've projected onto X, Y, Z axes. But what if we want to explore projections onto **arbitrary orthogonal axes**?

This helps understand:
- How PCA searches through all possible orientations
- Why certain axes capture more variance than others
- The connection between rotation and projection

### Euler Angles

We define a new coordinate system using three angles (α, β, γ):
- **α:** Rotation around X-axis
- **β:** Rotation around Y-axis  
- **γ:** Rotation around Z-axis

This creates three new orthogonal axes that we can project onto.

---

In [ ]:
def plot_euler_projections(var_x=5.0, var_y=2.0, var_z=1.0, 
                          rot_x=20, rot_y=10, rot_z=0,
                          alpha=0, beta=0, gamma=0):
    """
    Visualize projections onto arbitrary axes defined by Euler angles.
    """
    # Generate data
    data = generate_3d_data(var_x, var_y, var_z, rot_x, rot_y, rot_z)
    L = np.max(np.abs(data)) * 1.2
    
    # Compute new axes from Euler angles
    ax_alpha, ax_beta, ax_gamma = euler_axes(alpha, beta, gamma)

    # Create projections
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # α–β plane
    u, v = project_to_plane(data, ax_alpha, ax_beta)
    axes[0].scatter(u, v, alpha=0.5, c="red", s=10)
    axes[0].set_xlim(-L, L)
    axes[0].set_ylim(-L, L)
    axes[0].set_aspect("equal")
    axes[0].set_xlabel("α-axis")
    axes[0].set_ylabel("β-axis")
    axes[0].set_title("Projection: α–β plane")
    axes[0].grid(True, alpha=0.3)

    # β–γ plane
    u, v = project_to_plane(data, ax_beta, ax_gamma)
    axes[1].scatter(u, v, alpha=0.5, c="red", s=10)
    axes[1].set_xlim(-L, L)
    axes[1].set_ylim(-L, L)
    axes[1].set_aspect("equal")
    axes[1].set_xlabel("β-axis")
    axes[1].set_ylabel("γ-axis")
    axes[1].set_title("Projection: β–γ plane")
    axes[1].grid(True, alpha=0.3)

    # γ–α plane
    u, v = project_to_plane(data, ax_gamma, ax_alpha)
    axes[2].scatter(u, v, alpha=0.5, c="red", s=10)
    axes[2].set_xlim(-L, L)
    axes[2].set_ylim(-L, L)
    axes[2].set_aspect("equal")
    axes[2].set_xlabel("γ-axis")
    axes[2].set_ylabel("α-axis")
    axes[2].set_title("Projection: γ–α plane")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Sliders (reuse variance and rotation sliders from Module 3)
alpha_slider = FloatSlider(min=0, max=180, step=5, value=0, description="α")
beta_slider = FloatSlider(min=0, max=180, step=5, value=0, description="β")
gamma_slider = FloatSlider(min=0, max=180, step=5, value=0, description="γ")

out_euler = interactive_output(
    plot_euler_projections,
    {
        "var_x": varx_slider, "var_y": vary_slider, "var_z": varz_slider,
        "rot_x": rotx_slider, "rot_y": roty_slider, "rot_z": rotz_slider,
        "alpha": alpha_slider, "beta": beta_slider, "gamma": gamma_slider
    }
)

ui_euler = VBox([
    HTML("<h3>Custom Projection Axes (Euler Angles)</h3>"),
    HTML("<p>Adjust α, β, γ to explore projections onto arbitrary orthogonal axes</p>"),
    HBox([alpha_slider, beta_slider, gamma_slider]),
    out_euler
])

display(ui_euler)

## 🔍 Advanced Exploration

### Challenge: Find the "best" projection

1. Keep data parameters fixed: `Var X=5, Var Y=2, Var Z=1, Rot X=20, Rot Y=10`
2. Try different combinations of α, β, γ
3. **Question:** Which projection shows the most spread?
4. **Hint:** The "best" projection should align with the data's natural orientation

**This is exactly what PCA does automatically!** It searches through all possible rotations to find the axes that maximize spread (variance).

---

### Connection to PCA

If you adjust α, β, γ until the first projection (α–β plane) shows maximum spread, you've essentially found **PC1 and PC2** manually!

The mathematical algorithm (eigenvalue decomposition) does this systematically without trial and error.

---

# Module 5: Summary & Next Steps

## 🎓 What You've Learned

| Concept | Key Insight |
|---------|-------------|
| **Projection** | PCA projects data onto lower-dimensional spaces |
| **Variance Maximization** | PCA finds directions that preserve the most information |
| **Error Minimization** | Equivalent to minimizing reconstruction error |
| **Outlier Sensitivity** | PCA is sensitive to outliers due to variance formula |
| **Multiple Components** | In 3D, PCA finds 3 orthogonal principal components |
| **Arbitrary Axes** | PCA searches all possible rotations systematically |

---

## 🔗 Connecting to sklearn.PCA

Everything you explored can be done with one line of code:

In [8]:
from sklearn.decomposition import PCA

# Generate some 3D data
data_example = generate_3d_data(var_x=5, var_y=2, var_z=1, rot_x=20, rot_y=10)

# Apply PCA
pca = PCA(n_components=3)
scores = pca.fit_transform(data_example)
loadings = pca.components_

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("\nPrincipal component directions (loadings):")
print(loadings)
print("\n✓ PC1 captures the most variance!")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  PC3: {pca.explained_variance_ratio_[2]*100:.1f}%")

Explained variance ratio: [0.60669916 0.26888364 0.12441719]

Principal component directions (loadings):
[[ 0.98243783  0.02671742 -0.1846675 ]
 [ 0.04396119  0.92869208  0.36823694]
 [ 0.18133759 -0.3698881   0.9112077 ]]

✓ PC1 captures the most variance!
  PC1: 60.7%
  PC2: 26.9%
  PC3: 12.4%


---

## 📚 Next Steps

### For Real Data Applications:
1. **PCA_CityTemp_Plots.ipynb** — Comprehensive PCA on European city temperature data:
   - Score plots, loading plots, biplots
   - Outlier detection (Hotelling's T², Q-statistic)
   - Contribution analysis
   - Model validation

2. **ComprehensivePCA.ipynb** — Advanced techniques:
   - Feature importance
   - Leverage and influence
   - Residual analysis
   - Cross-validation

### For More Theory:
- Study **eigenvalue decomposition** and **singular value decomposition (SVD)**
- Learn about **kernel PCA** for non-linear dimensionality reduction
- Explore **robust PCA** methods for outlier-heavy data

---

## ✅ Final Self-Assessment

Can you explain these to a classmate?

- [ ] Why minimizing projection error equals maximizing variance
- [ ] How to interpret principal components geometrically
- [ ] Why PCA is sensitive to outliers
- [ ] The difference between scores and loadings
- [ ] When to use 2 vs 3 principal components

If you checked all boxes, you're ready for real-world PCA! 🎉

---

**Questions or feedback?** Add them to the course discussion forum or ask your instructor.

**Happy analyzing!** 📊✨